# Prospective source and forecast registry

This workflow records received data and prepares monthly forecast records for October 2026–September 2027. **The final hybrid was fitted and frozen on 29 September 2026; no real forecast has been issued.** ERA5, SEAS5 and the four required ocean indices passed their source checks. October emission remains blocked by missing CFSv2 members. See the [operational record](OPERATIONAL_STATUS.md) and [fitting and inference workflow](BASELINE_RUN.md).

The [notebook](prospective_registry.ipynb) collects public sources, audits inputs, restores records and reports readiness. It does not train or substitute a model when inputs are unavailable.

## Plan and checks

Atmosphere T−4; ocean indices T−3; seasonal initialization T−1. SST T−2 is archived for research only. Final training is fixed to January 1993–December 2022 with the research lags. Old fold models or competition models with different lags cannot substitute for that package.

Checks cover actual UTC receipt time, hashes, required months, units, grids, ensemble completeness, parent files, model-package completeness, deadline, duplicate emission and forecast validity. Unknown first-publication time remains unknown. Missing sources block emission. Forecast and training climatology are frozen together; later ERA5 revisions remain separate verification records.

The local hash chain is not an independent timestamp or tamper-proof store. Reports retain months without forecasts instead of omitting them.

## Run

Use Python 3.12 and [requirements.txt](requirements.txt):

```bash
python research/prospective/test_registry.py
python research/prospective/collect_sources.py --alvo 2026-10 --cfsv2
python research/prospective/prepare_cfsv2.py --alvo 2026-10
```

`--alvo` means target month; `--atualizar` on the preparation command requests a fresh source version. Compatibility keys remain stable. Commands run once; there is no automatic schedule.

Save outputs. Attach the complete previous registry in Kaggle to restore its chain and objects. Restarting without it creates another chain and cannot prove earlier acquisitions.

On 28 and 29 September 2026, the acquired September CFSv2 fields lacked members 21–24 and were blocked without a partial mean. A later acquisition must be checked again; this record does not assert the source's future state. Sources are listed in [DATA.md](../../docs/DATA.md).


In [ ]:
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RAIZ = BASE/'rainfall_prospective'
CODIGO = RAIZ/'code'
CODIGO.mkdir(parents=True, exist_ok=True)
BUNDLE = {'README.md': "# Prospective source and forecast registry\n\nThis workflow records received data and prepares monthly forecast records for October 2026–September 2027. **The final hybrid was fitted and frozen on 29 September 2026; no real forecast has been issued.** ERA5, SEAS5 and the four required ocean indices passed their source checks. October emission remains blocked by missing CFSv2 members. See the [operational record](OPERATIONAL_STATUS.md) and [fitting and inference workflow](BASELINE_RUN.md).\n\nThe [notebook](prospective_registry.ipynb) collects public sources, audits inputs, restores records and reports readiness. It does not train or substitute a model when inputs are unavailable.\n\n## Plan and checks\n\nAtmosphere T−4; ocean indices T−3; seasonal initialization T−1. SST T−2 is archived for research only. Final training is fixed to January 1993–December 2022 with the research lags. Old fold models or competition models with different lags cannot substitute for that package.\n\nChecks cover actual UTC receipt time, hashes, required months, units, grids, ensemble completeness, parent files, model-package completeness, deadline, duplicate emission and forecast validity. Unknown first-publication time remains unknown. Missing sources block emission. Forecast and training climatology are frozen together; later ERA5 revisions remain separate verification records.\n\nThe local hash chain is not an independent timestamp or tamper-proof store. Reports retain months without forecasts instead of omitting them.\n\n## Run\n\nUse Python 3.12 and [requirements.txt](requirements.txt):\n\n```bash\npython research/prospective/test_registry.py\npython research/prospective/collect_sources.py --alvo 2026-10 --cfsv2\npython research/prospective/prepare_cfsv2.py --alvo 2026-10\n```\n\n`--alvo` means target month; `--atualizar` on the preparation command requests a fresh source version. Compatibility keys remain stable. Commands run once; there is no automatic schedule.\n\nSave outputs. Attach the complete previous registry in Kaggle to restore its chain and objects. Restarting without it creates another chain and cannot prove earlier acquisitions.\n\nOn 28 and 29 September 2026, the acquired September CFSv2 fields lacked members 21–24 and were blocked without a partial mean. A later acquisition must be checked again; this record does not assert the source's future state. Sources are listed in [DATA.md](../../docs/DATA.md).\n", 'registry.py': '"""Auditable local registry; not a signature or independent timestamp."""\nfrom __future__ import annotations\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport re\n\nclass ContratoError(ValueError):\n    pass\n\ndef exigir(ok, mensagem):\n    if not ok:\n        raise ContratoError(mensagem)\n\ndef agora():\n    return datetime.now(timezone.utc)\n\ndef instante(valor):\n    t = datetime.fromisoformat(str(valor).replace(\'Z\', \'+00:00\'))\n    exigir(t.tzinfo is not None, \'Timestamp has no timezone.\')\n    return t.astimezone(timezone.utc)\n\ndef mes(valor):\n    exigir(bool(re.fullmatch(\'\\\\d{4}-\\\\d{2}\', str(valor))), \'Month must use YYYY-MM.\')\n    t = datetime.strptime(valor, \'%Y-%m\').replace(tzinfo=timezone.utc)\n    return t\n\ndef deslocar(valor, n):\n    t = mes(valor)\n    k = t.year * 12 + t.month - 1 + n\n    return f\'{k // 12:04d}-{k % 12 + 1:02d}\'\n\ndef canonico(obj):\n    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(\',\', \':\'), allow_nan=False).encode(\'utf-8\')\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n# Exclusive creation prevents silent replacement of a recorded artifact.\ndef escrever_novo(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'xb\') as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n\nclass Registro:\n\n    def __init__(self, pasta):\n        self.pasta = Path(pasta).resolve()\n        self.pasta.mkdir(parents=True, exist_ok=True)\n        for nome in (\'objetos\', \'eventos\'):\n            (self.pasta / nome).mkdir(exist_ok=True)\n\n    @contextmanager\n    def transacao(self):\n        lock = self.pasta / \'.escrita.lock\'\n        try:\n            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        except FileExistsError:\n            raise ContratoError(\'Registry is in use or has a lock from an interrupted run. Do not remove it automatically.\') from None\n        os.close(fd)\n        try:\n            yield\n        finally:\n            lock.unlink()\n\n    def guardar(self, dados):\n        h = digest(dados)\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        if p.exists():\n            exigir(p.read_bytes() == dados, \'Registry object changed.\')\n        else:\n            try:\n                escrever_novo(p, dados)\n            except FileExistsError:\n                exigir(p.read_bytes() == dados, \'Object collision.\')\n        return dict(sha256=h, bytes=len(dados))\n\n    def ler_objeto(self, ref):\n        h = ref[\'sha256\']\n        exigir(bool(re.fullmatch(\'[0-9a-f]{64}\', h)), \'Invalid hash.\')\n        p = self.pasta / \'objetos\' / (h + \'.bin\')\n        exigir(p.is_file(), \'Missing object: \' + h)\n        b = p.read_bytes()\n        exigir(len(b) == ref[\'bytes\'] and digest(b) == h, \'Modified object: \' + h)\n        return b\n\n    def eventos(self, conferir_objetos=False):\n        out, anterior, ultimo = ([], None, None)\n        for i, p in enumerate(sorted((self.pasta / \'eventos\').glob(\'*.json\')), 1):\n            e = json.loads(p.read_text(encoding=\'utf-8\'))\n            h = e.pop(\'id\')\n            exigir(digest(canonico(e)) == h, \'Event changed: \' + p.name)\n            exigir(p.name == f\'{i:08d}_{h}.json\', \'Incomplete event sequence.\')\n            exigir(e[\'sequencia\'] == i and e[\'anterior\'] == anterior, \'Invalid event chain.\')\n            t = instante(e[\'registrado_em_utc\'])\n            exigir(ultimo is None or t >= ultimo, \'Clock moved backwards; forecast blocked.\')\n            e[\'id\'] = h\n            if conferir_objetos:\n                for ref in e[\'dados\'].get(\'objetos\', []):\n                    self.ler_objeto(ref)\n            out.append(e)\n            anterior, ultimo = (h, t)\n        return out\n\n    def _adicionar(self, tipo, dados, prazo_exclusivo=None):\n        existentes = self.eventos()\n        t = agora()\n        exigir(not existentes or t >= instante(existentes[-1][\'registrado_em_utc\']), \'Clock moved backwards.\')\n        exigir(prazo_exclusivo is None or t < prazo_exclusivo, \'Issue deadline passed before the event could be registered.\')\n        e = dict(sequencia=len(existentes) + 1, anterior=existentes[-1][\'id\'] if existentes else None, registrado_em_utc=t.isoformat(), tipo=tipo, dados=dados)\n        e[\'id\'] = digest(canonico(e))\n        escrever_novo(self.pasta / \'eventos\' / f\'{e[\'sequencia\']:08d}_{e[\'id\']}.json\', canonico(e))\n        return e\n\n    def adicionar(self, tipo, dados):\n        with self.transacao():\n            return self._adicionar(tipo, dados)\n\n    def obter(self, ident, tipo=None):\n        encontrados = [e for e in self.eventos() if e[\'id\'] == ident]\n        exigir(len(encontrados) == 1, \'Unknown event.\')\n        e = encontrados[0]\n        exigir(tipo is None or e[\'tipo\'] == tipo, \'Incorrect event type.\')\n        return e\n\n    def iniciar(self, plano):\n        with self.transacao():\n            anteriores = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n            if anteriores:\n                exigir(len(anteriores) == 1 and anteriores[0][\'dados\'][\'plano\'] == plano, \'Plan changed. Start a separate registry and experiment.\')\n                return anteriores[0]\n            exigir(agora() < mes(plano[\'primeiro_mes\']), \'The protocol cannot start with a past or ongoing month.\')\n            return self._adicionar(\'protocolo\', dict(plano=plano, objetos=[]))\n\n    def protocolo(self):\n        e = [e for e in self.eventos() if e[\'tipo\'] == \'protocolo\']\n        exigir(len(e) == 1, \'Initialize the protocol first.\')\n        return e[0]\n\n    def recibo(self, fonte, dados, metadados, transporte):\n        """Use the actual write time, never file mtime or a nominal monthly date."""\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Source outside the protocol.\')\n        with self.transacao():\n            ref = self.guardar(dados)\n            return self._adicionar(\'fonte_recebida\', dict(fonte=fonte, objeto=ref, objetos=[ref], inspecao=metadados, transporte=transporte, primeira_publicacao_utc=None, evidencia=\'bytes received during this acquisition; not proof of first historical publication\'))\n\n    def modelo(self, arquivos, auditoria):\n        """Register the final model package; this does not train or certify the algorithm."""\n        p = self.protocolo()\n        plano = p[\'dados\'][\'plano\']\n        exigir(set(arquivos) == set(plano[\'arquivos_modelo\']), \'Incomplete or unexpected model package.\')\n        exigir(auditoria[\'treino_inicio\'] == plano[\'treino_inicio\'] and auditoria[\'treino_fim\'] == plano[\'treino_fim\'], \'Training differs from the plan.\')\n        exigir(auditoria.get(\'protocolo_id\') == p[\'id\'], \'Audit does not match the protocol.\')\n        with self.transacao():\n            exigir(not any((e[\'tipo\'] == \'modelo_congelado\' for e in self.eventos())), \'A model is already frozen for this experiment.\')\n            refs = {nome: self.guardar(Path(path).read_bytes()) for nome, path in arquivos.items()}\n            return self._adicionar(\'modelo_congelado\', dict(protocolo=p[\'id\'], arquivos=refs, objetos=list(refs.values()), auditoria=auditoria))\n\n    def derivado(self, fonte, dados, metadados, pais, codigo):\n        exigir(bool(pais) and len(pais) == len(set(pais)), \'Missing or duplicate parent files.\')\n        exigir(fonte in self.protocolo()[\'dados\'][\'plano\'][\'fontes\'], \'Source outside the protocol.\')\n        with self.transacao():\n            for ident in pais:\n                e = self.obter(ident, \'fonte_recebida\')\n                exigir(e[\'dados\'][\'fonte\'] == fonte, \'Parent belongs to a different source.\')\n                self.ler_objeto(e[\'dados\'][\'objeto\'])\n            ref = self.guardar(dados)\n            cod = self.guardar(Path(codigo).read_bytes())\n            return self._adicionar(\'fonte_recebida\', dict(fonte=fonte, objeto=ref, objetos=[ref, cod], inspecao=metadados, pais=pais, codigo=cod, primeira_publicacao_utc=None, transporte=dict(metodo=\'local transformation of registered parent files\'), evidencia=\'parent acquisition and processing with preserved adapter code\'))\n\n    def prontidao(self, alvo):\n        mes(alvo)\n        protocolo = self.protocolo()\n        p = protocolo[\'dados\'][\'plano\']\n        exigir(p[\'primeiro_mes\'] <= alvo <= p[\'ultimo_mes\'], \'Month outside the monitoring period.\')\n        ev = self.eventos(conferir_objetos=True)\n        fontes, faltantes = ({}, [])\n        for fonte, regra in p[\'fontes\'].items():\n            if not regra[\'obrigatoria\']:\n                continue\n            esperado = deslocar(alvo, -regra[\'lag\'])\n            candidatos = [e for e in ev if e[\'tipo\'] == \'fonte_recebida\' and e[\'dados\'][\'fonte\'] == fonte and (esperado in e[\'dados\'][\'inspecao\'].get(\'meses\', [])) and (e[\'dados\'][\'inspecao\'].get(\'validado\') is True) and (instante(e[\'registrado_em_utc\']) < mes(alvo))]\n            if candidatos:\n                fontes[fonte] = candidatos[-1][\'id\']\n            else:\n                faltantes.append(f\'{fonte}: month {esperado}\')\n        modelos = [e for e in ev if e[\'tipo\'] == \'modelo_congelado\']\n        if not modelos:\n            faltantes.append(\'model: final package for the lagged-source baseline has not been registered\')\n        if agora() >= mes(alvo):\n            faltantes.append(\'deadline: the month has started; backdating is prohibited\')\n        emitida = any(e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo for e in ev)\n        return dict(mes_alvo=alvo, pronto=not faltantes, faltantes=faltantes, fontes=fontes, modelo=modelos[0][\'id\'] if modelos else None, protocolo=protocolo[\'id\'], previsao_emitida=emitida)\n\n    def congelar(self, alvo, arquivo, inferencia):\n        """Accept audited inference only; no backdating or overwriting."""\n        from data_contracts import validar_previsao\n        with self.transacao():\n            exigir(not any((e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo for e in self.eventos())), \'A prediction already exists; revisions require a separate series.\')\n            status = self.prontidao(alvo)\n            exigir(status[\'pronto\'], \'Forecast blocked: \' + \'; \'.join(status[\'faltantes\']))\n            exigir(inferencia.get(\'modelo\') == status[\'modelo\'] and inferencia.get(\'fontes\') == status[\'fontes\'] and (inferencia.get(\'protocolo\') == status[\'protocolo\']), \'Inference did not use the selected input versions.\')\n            b = Path(arquivo).read_bytes()\n            exigir(inferencia.get(\'previsao_sha256\') == digest(b), \'Inference hash differs from the file.\')\n            resumo = validar_previsao(arquivo, alvo)\n            ref = self.guardar(b)\n            exigir(agora() < mes(alvo), \'Deadline passed during preparation.\')\n            return self._adicionar(\'previsao_congelada\', dict(mes_alvo=alvo, protocolo=status[\'protocolo\'], modelo=status[\'modelo\'], fontes=status[\'fontes\'], inferencia=inferencia, previsao=ref, objetos=[ref], validacao=resumo, prazo_exclusivo_utc=mes(alvo).isoformat(), carimbo_independente=False), prazo_exclusivo=mes(alvo))\n\n    def exportar_resumo(self, destino):\n        ev = self.eventos(conferir_objetos=True)\n        d = dict(eventos=len(ev), ponta=ev[-1][\'id\'] if ev else None, conferido_em_utc=agora().isoformat(), garantia=\'local event chain and hashes; not a signature or tamper-proof storage\', fontes=[dict(id=e[\'id\'], recebido_em_utc=e[\'registrado_em_utc\'], **e[\'dados\']) for e in ev if e[\'tipo\'] == \'fonte_recebida\'], previsoes=[e for e in ev if e[\'tipo\'] == \'previsao_congelada\'])\n        Path(destino).write_text(json.dumps(d, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n        return d\n', 'data_contracts.py': '"""Inspect acquired bytes and validate forecast field contracts."""\nfrom pathlib import Path\nimport tempfile\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom registry import exigir, ContratoError, mes\nLAT = np.arange(-60, 15.001, 0.25)\nLON = np.arange(-90, -24.999, 0.25)\n\ndef indices_psl(dados):\n    linhas = dados.decode(\'utf-8-sig\').strip().splitlines()\n    exigir(len(linhas) > 3 and len(linhas[0].split()) == 2, \'Invalid PSL header.\')\n    a, b = [int(x) for x in linhas[0].split()]\n    exigir(1800 <= a <= b <= 2200, \'Invalid declared years.\')\n    exigir(len(linhas) > b - a + 2, \'Incomplete PSL series.\')\n    matriz = []\n    for ano, linha in zip(range(a, b + 1), linhas[1:b - a + 2]):\n        v = linha.split()\n        exigir(len(v) == 13 and int(v[0]) == ano, \'Invalid PSL years or months.\')\n        matriz.extend(((f\'{ano:04d}-{m:02d}\', float(x)) for m, x in enumerate(v[1:], 1)))\n    sentinela = float(linhas[b - a + 2].strip())\n    exigir(sentinela < -90, \'Unexpected missing-value marker.\')\n    pares = [(t, v) for t, v in matriz if v != sentinela]\n    exigir(pares and all((np.isfinite(v) and abs(v) < 20 for _, v in pares)), \'Index outside the expected range.\')\n    return dict(validado=True, formato=\'monthly NOAA PSL\', unidades=\'degC\', meses=[t for t, _ in pares], ultimo_mes=pares[-1][0], ausentes=len(matriz) - len(pares), rodape=\'\\n\'.join(linhas[b - a + 3:]))\n\ndef inspecionar_netcdf(dados, perfil, esperado=None):\n    exigir(dados[:3] == b\'CDF\' or dados[:8] == b\'\\x89HDF\\r\\n\\x1a\\n\', \'Response is not NetCDF.\')\n    with tempfile.TemporaryDirectory() as td:\n        path = Path(td) / \'entrada.nc\'\n        path.write_bytes(dados)\n        with xr.open_dataset(path, decode_times=perfil != \'cfsv2_bruto\') as ds:\n            if perfil == \'ersstv5\':\n                exigir(ds.attrs.get(\'product_version\') == \'Version v5\', \'Different ERSST version.\')\n                exigir(\'sst\' in ds and ds.sst.attrs.get(\'units\') == \'degree_C\', \'Different SST variable or units.\')\n                ts = pd.DatetimeIndex(ds.time.values)\n                meses = ts.strftime(\'%Y-%m\').tolist()\n                exigir(len(meses) == 1 and meses[0] == esperado, \'SST month does not match the request.\')\n                exigir(ds.sizes.get(\'lat\') == 89 and ds.sizes.get(\'lon\') == 180, \'Unexpected ERSST grid.\')\n                exigir(np.array_equal(ds.lat, np.arange(-88, 90, 2)) and np.array_equal(ds.lon, np.arange(0, 360, 2)), \'Unexpected ERSST coordinates.\')\n                vals = ds.sst.values\n                f = vals[np.isfinite(vals)]\n                exigir(f.size > 1000 and (not np.isinf(vals).any()) and (f.min() >= -1.801) and (f.max() <= 45), \'Invalid SST field.\')\n                return dict(validado=True, meses=meses, versao=\'ERSSTv5\', unidades=\'degree_C\', dimensoes=dict(ds.sizes), minimo=float(f.min()), maximo=float(f.max()))\n            exigir(perfil == \'cfsv2_bruto\', \'Unknown inspection profile.\')\n            return dict(validado=False, meses=[], dimensoes=dict(ds.sizes), variaveis=list(ds.data_vars), pendencia=\'Audit real initialization dates, members, lead 1.5, units and grid before inference.\')\n\ndef validar_previsao(path, alvo):\n    with xr.open_dataset(path) as ds:\n        exigir(set(ds.data_vars) == {\'precipitacao\', \'climatologia\'}, \'Both forecast and control climatology are required.\')\n        exigir(set(ds.dims) == {\'time\', \'lat\', \'lon\'} and ds.sizes[\'time\'] == 1, \'Invalid dimensions.\')\n        exigir(np.array_equal(ds.lat, LAT) and np.array_equal(ds.lon, LON), \'Forecast grid differs from the reference.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.DatetimeIndex([mes(alvo).replace(tzinfo=None)])), \'Incorrect forecast target.\')\n        for n in ds.data_vars:\n            a = ds[n]\n            exigir(a.dims == (\'time\', \'lat\', \'lon\') and a.attrs.get(\'units\') == \'mm/day\', \'Invalid output dimension order or units.\')\n            exigir(np.isfinite(a.values).all() and (a.values >= 0).all(), \'Prediction contains missing or negative values.\')\n        return dict(pontos=int(ds.precipitacao.size), minimo=float(ds.precipitacao.min()), maximo=float(ds.precipitacao.max()), unidades=\'mm/day\')\n\ndef metricas(previsao, verdade):\n    """Require exact alignment; do not mask observation, grid or unit errors."""\n    with xr.open_dataset(previsao) as a, xr.open_dataset(verdade) as b:\n        exigir(set(b.data_vars) == {\'tp\'}, \'Observation must contain only tp.\')\n        exigir(b.tp.attrs.get(\'units\') == \'mm/day\' and b.tp.dims == (\'time\', \'lat\', \'lon\'), \'Invalid observation contract.\')\n        exigir(np.isfinite(b.tp.values).all() and (b.tp.values >= 0).all(), \'Incomplete or invalid observation.\')\n        linhas = []\n        for nome in (\'precipitacao\', \'climatologia\'):\n            p, y = xr.align(a[nome], b.tp, join=\'exact\')\n            exigir(p.shape == y.shape, \'Observation does not cover the complete prediction.\')\n            erro = np.asarray(p.values, dtype=np.float64) - np.asarray(y.values, dtype=np.float64)\n            area = np.broadcast_to(np.cos(np.deg2rad(y.lat.values))[None, :, None], erro.shape)\n            for regiao, mask in [(\'dominio\', np.ones(len(y.lat), dtype=bool)), (\'sul_35\', y.lat.values < -35), (\'35_a_15\', (y.lat.values >= -35) & (y.lat.values < -15)), (\'norte_15\', y.lat.values >= -15)]:\n                e, w = (erro[:, mask, :], area[:, mask, :])\n                exigir(e.size > 0, \'Empty region.\')\n                sse, n = (float((e * e).sum()), int(e.size))\n                linhas.append(dict(modelo=nome, regiao=regiao, n=n, sse=sse, soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), rmse=float(np.sqrt(sse / n)), mae=float(np.abs(e).mean()), vies=float(e.mean()), sse_area=float((e * e * w).sum()), peso_area=float(w.sum()), rmse_area=float(np.sqrt((e * e * w).sum() / w.sum()))))\n        return linhas\n\ndef conferir_observacao_bruta(bruto, normalizado, alvo):\n    """Validate ERA5 monthly conversion; reject unknown format or expver."""\n    with xr.open_dataset(bruto) as ds, xr.open_dataset(normalizado) as n:\n        exigir(\'tp\' in ds and set(n.data_vars) == {\'tp\'}, \'Missing rainfall variable tp.\')\n        a = ds.tp\n        exigir(a.attrs.get(\'units\') in {\'m\', \'m/day\', \'m day**-1\'}, \'Unexpected raw ERA5 units.\')\n        stream = a.attrs.get(\'GRIB_stream\', ds.attrs.get(\'GRIB_stream\'))\n        exigir(stream == \'moda\', \'Require monthly means of daily means, stream moda; do not infer conversion for a different product.\')\n        if \'expver\' in a.dims:\n            exigir(a.sizes[\'expver\'] == 1 and int(a.expver.values[0]) == 1, \'ERA5T or mixed versions are not accepted.\')\n            a = a.isel(expver=0, drop=True)\n        elif \'expver\' in ds:\n            exigir(np.asarray(ds.expver).size == 1 and int(np.asarray(ds.expver).ravel()[0]) == 1, \'Nonfinal expver.\')\n        else:\n            ver = a.attrs.get(\'GRIB_experimentVersionNumber\', ds.attrs.get(\'GRIB_experimentVersionNumber\'))\n            exigir(str(ver) in {\'1\', \'0001\'}, \'Raw file contains no evidence of final expver.\')\n        a = a.rename({x: y for x, y in [(\'valid_time\', \'time\'), (\'latitude\', \'lat\'), (\'longitude\', \'lon\')] if x in a.dims})\n        exigir(set(a.dims) == {\'time\', \'lat\', \'lon\'} and a.sizes[\'time\'] == 1, \'Unexpected raw target dimensions.\')\n        ts = pd.DatetimeIndex(a.time.values)\n        exigir(ts[0].day == 1 and ts.strftime(\'%Y-%m\').tolist() == [alvo], \'Raw month differs from the target.\')\n        a = a.assign_coords(time=[np.datetime64(alvo + \'-01\')], lon=(a.lon + 180) % 360 - 180).sortby(\'lat\').sortby(\'lon\')\n        a = a.transpose(\'time\', \'lat\', \'lon\')\n        a, b = xr.align(a, n.tp, join=\'exact\')\n        exigir(a.shape == b.shape and np.allclose(a.values * 1000, b.values, rtol=1e-06, atol=1e-06), \'Target conversion does not match raw ERA5 multiplied by 1000.\')\n        return dict(stream=\'moda\', expver=1, conversao=\'m * 1000 -> mm/day\', mes_alvo=alvo)\n', 'collect_sources.py': '"""Acquire public sources as separate versions without publishing or issuing a forecast."""\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport argparse\nimport json\nimport requests\nfrom registry import Registro, agora, deslocar, exigir\nfrom data_contracts import indices_psl, inspecionar_netcdf\nFONTES = {\'nino34\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'nino12\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'tna\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'tsa\': \'https://psl.noaa.gov/data/correlation/tsa.data\'}\nHOSTS = {\'psl.noaa.gov\', \'www.ncei.noaa.gov\', \'iridl.ldeo.columbia.edu\'}\n\ndef baixar(registro, fonte, url, perfil, esperado=None):\n    exigir(urlparse(url).scheme == \'https\' and urlparse(url).hostname in HOSTS, "Use the producer\'s authorized HTTPS endpoint.")\n    inicio = agora().isoformat()\n    try:\n        with requests.Session() as s:\n            s.headers[\'User-Agent\'] = \'WorCAP-rainfall-research/1.0 (prospective-data-capture)\'\n            with s.get(url, stream=True, timeout=(20, 90), allow_redirects=False) as r:\n                exigir(r.status_code == 200, f\'HTTP {r.status_code}\')\n                partes, tamanho = ([], 0)\n                for chunk in r.iter_content(128 * 1024):\n                    tamanho += len(chunk)\n                    exigir(tamanho <= 32 * 1024 * 1024, \'Response exceeds the 32 MiB limit.\')\n                    partes.append(chunk)\n                b = b\'\'.join(partes)\n                exigir(bool(b), \'Empty response.\')\n                try:\n                    meta = indices_psl(b) if perfil == \'psl\' else inspecionar_netcdf(b, perfil, esperado)\n                except Exception as erro_validacao:\n                    meta = dict(validado=False, meses=[], erro_inspecao=type(erro_validacao).__name__, pendencia=\'Response preserved in quarantine; do not use for inference.\')\n                transporte = dict(url=url, metodo=\'HTTPS GET\', iniciado_em_utc=inicio, resposta_em_utc=agora().isoformat(), status_http=r.status_code, cabecalhos={k: r.headers[k] for k in [\'Date\', \'Last-Modified\', \'ETag\', \'Content-Type\', \'Content-Length\'] if k in r.headers}, last_modified_nao_e_primeira_publicacao=True)\n        e = registro.recibo(fonte, b, meta, transporte)\n        print(f\'{fonte}: received; validated={meta[\'validado\']}; {len(b):,} bytes; id={e[\'id\'][:12]}\', flush=True)\n        return e\n    except Exception as erro:\n        registro.adicionar(\'coleta_falhou\', dict(fonte=fonte, url=url, iniciado_em_utc=inicio, erro_tipo=type(erro).__name__, objetos=[]))\n        print(f\'{fonte}: acquisition rejected ({type(erro).__name__}). No substitution with data from a different month.\', flush=True)\n        return None\n\ndef executar(pasta, plano_path, alvo, incluir_cfsv2=False):\n    r = Registro(pasta)\n    plano = json.loads(Path(plano_path).read_text(encoding=\'utf-8\'))\n    r.iniciar(plano)\n    r.prontidao(alvo)\n    saidas = [baixar(r, n, u, \'psl\') for n, u in FONTES.items()]\n    t = deslocar(alvo, -2).replace(\'-\', \'\')\n    u = f\'https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/ersst.v5.{t}.nc\'\n    saidas.append(baixar(r, \'ersstv5\', u, \'ersstv5\', deslocar(alvo, -2)))\n    if incluir_cfsv2:\n        origem = deslocar(alvo, -1)\n        ano, m = map(int, origem.split(\'-\'))\n        mon = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[m - 1]\n        data = f\'(0000%201%20{mon}%20{ano})\'\n        raiz = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n        u = raiz + f\'S/{data}/{data}/RANGE/L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\'\n        saidas.append(baixar(r, \'cfsv2\', u, \'cfsv2_bruto\', alvo))\n    r.exportar_resumo(Path(pasta) / \'resumo_registro.json\')\n    estado = r.prontidao(alvo)\n    (Path(pasta) / \'prontidao.json\').write_text(json.dumps(estado, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    print(json.dumps(estado, indent=2, ensure_ascii=False), flush=True)\n    return estado\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registro\', default=\'outputs/prospective\')\n    p.add_argument(\'--plano\', default=str(Path(__file__).with_name(\'plan.json\')))\n    p.add_argument(\'--alvo\', required=True)\n    p.add_argument(\'--cfsv2\', action=\'store_true\')\n    a = p.parse_args()\n    executar(a.registro, a.plano, a.alvo, a.cfsv2)\n', 'prepare_cfsv2.py': '"""Acquire and audit next-month CFSv2 predictions while preserving raw inputs."""\nfrom pathlib import Path\nimport tempfile\nimport pandas as pd\nfrom registry import Registro, deslocar, exigir\nfrom collect_sources import baixar\nimport cfsv2_adapter as a\n\ndef preparar(registro, alvo, atualizar=False):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    origem = deslocar(alvo, -1)\n    ano, mm = map(int, origem.split(\'-\'))\n    m = (\'Jan\', \'Feb\', \'Mar\', \'Apr\', \'May\', \'Jun\', \'Jul\', \'Aug\', \'Sep\', \'Oct\', \'Nov\', \'Dec\')[mm - 1]\n    d = f\'(0000%201%20{m}%20{ano})\'\n    intervalo = f\'S/{d}/{d}/RANGE/\'\n    urls = [a.FONTE + intervalo + \'L/1.5/VALUES/Y/-61/16/RANGE/X/269/336/RANGE/data.nc\', \'https://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NCEP/.EMC/.CFSv2/.REALTIME_ENSEMBLE/.FLXF/sampleS/\' + intervalo + \'data.nc\']\n    recibos = []\n    for u in urls:\n        antigos = [e for e in r.eventos() if e[\'tipo\'] == \'fonte_recebida\' and e[\'dados\'][\'fonte\'] == \'cfsv2\' and (e[\'dados\'][\'transporte\'].get(\'url\') == u)]\n        e = antigos[-1] if antigos and (not atualizar) else baixar(r, \'cfsv2\', u, \'cfsv2_bruto\', alvo)\n        exigir(e is not None, \'CFSv2 was not acquired; keep the forecast blocked.\')\n        recibos.append(e)\n    with tempfile.TemporaryDirectory() as td:\n        paths = []\n        for i, e in enumerate(recibos):\n            p = Path(td) / f\'original_{i}.nc\'\n            p.write_bytes(r.ler_objeto(e[\'dados\'][\'objeto\']))\n            paths.append(p)\n        ts = pd.DatetimeIndex([origem + \'-01\'])\n        tabela = a.tabela_inicializacoes(paths[1], ts)\n        try:\n            ds, auditoria = a.processar_bloco(paths[0], ts, tabela)\n        except ValueError as erro:\n            campos = a.carregar_membros(paths[0], ts)\n            import numpy as np\n            nvalidos = np.isfinite(campos[0]).sum(axis=(-1, -2))\n            info = dict(fonte=\'cfsv2\', mes_origem=origem, mes_alvo=alvo, motivo=str(erro), pais=[e[\'id\'] for e in recibos], pixels_finitos_por_membro={str(i + 1): int(n) for i, n in enumerate(nvalidos)}, membros_esperados=(np.flatnonzero(tabela[ts[0]][\'mascara\']) + 1).tolist(), objetos=[], acao=\'blocked; no exception or partial ensemble mean was applied\')\n            r.adicionar(\'auditoria_falhou\', info)\n            print(\'CFSv2 blocked:\', info[\'motivo\'], flush=True)\n            return None\n        exigir(pd.DatetimeIndex(ds.time.values).strftime(\'%Y-%m\').tolist() == [alvo], \'Incorrect CFSv2 target.\')\n        p = Path(td) / \'cfsv2.nc\'\n        ds.to_netcdf(p)\n        e = r.derivado(\'cfsv2\', p.read_bytes(), dict(validado=True, meses=[origem], meses_alvo=[alvo], unidades=\'mm/day\', lead=1.5, auditoria=auditoria), [e[\'id\'] for e in recibos], Path(a.__file__))\n        print(\'CFSv2 audited:\', origem, \'→\', alvo, \';\', auditoria[0][\'membros\'], \'members; id\', e[\'id\'][:12], flush=True)\n        return e\nif __name__ == \'__main__\':\n    import argparse\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--registro\', default=\'outputs/prospective\')\n    p.add_argument(\'--alvo\', required=True)\n    p.add_argument(\'--atualizar\', action=\'store_true\', help=\'Query the source again and preserve the new input version.\')\n    args = p.parse_args()\n    preparar(args.registro, args.alvo, args.atualizar)\n', 'cfsv2_adapter.py': '"""CFSv2 audit: actual dates, members, lead time, grid and units."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport cftime\nLEAD = 1.5\nLAT = np.arange(-61, 17, dtype=np.float32)\nLON_360 = np.arange(269, 337, dtype=np.float32)\nEXCECOES_MEMBROS = {}\nPOLITICA_MEMBROS = \'todos_membros_esperados_sem_lacunas_v1\'\nSCHEMA = \'nimbus_cfsv2_prospectivo_v1\'\nFONTE = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\n\ndef exigir(ok, mensagem):\n    if not bool(ok):\n        raise ValueError(mensagem)\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for parte in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(parte)\n    return h.hexdigest()\n\ndef meses_de_S(s):\n    exigir(str(s.attrs.get(\'units\', \'\')).strip() == \'months since 1960-01-01\', \'Unexpected S units.\')\n    exigir(str(s.attrs.get(\'calendar\')) in (\'360\', \'360_day\'), \'Unexpected S calendar.\')\n    valores = np.asarray(s.values, dtype=np.float64)\n    exigir(valores.ndim == 1 and np.isfinite(valores).all(), \'Invalid S.\')\n    exigir(np.equal(valores, np.rint(valores)).all(), \'S contains a fractional month.\')\n    return pd.DatetimeIndex([(pd.Period(\'1960-01\', freq=\'M\') + int(v)).to_timestamp() for v in valores])\n\ndef tabela_inicializacoes(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'sampleS\'} and set(ds.sampleS.dims) == {\'S\', \'M\'}, \'Unexpected sampleS table format.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'sampleS coverage mismatch.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'Unexpected table member IDs.\')\n    exigir(ds.sampleS.attrs.get(\'units\') == \'days since 1960-01-01\', \'Unexpected sampleS units.\')\n    exigir(str(ds.sampleS.attrs.get(\'calendar\')) in (\'365\', \'365_day\', \'noleap\'), \'Unexpected sampleS calendar.\')\n    resultado = {}\n    for k, origem in enumerate(origens):\n        numeros = ds.sampleS.transpose(\'S\', \'M\').values[k].astype(np.float64)\n        exigir(not np.isinf(numeros).any(), \'Infinite initialization date.\')\n        mascara = np.isfinite(numeros)\n        n = 28 if origem.month == 11 else 24\n        exigir(np.array_equal(mascara, np.arange(28) < n), f\'Member table mismatch in {origem.date()}: expected {n}.\')\n        datas_cf = cftime.num2date(numeros[mascara], \'days since 1960-01-01\', calendar=\'365_day\')\n        datas = pd.DatetimeIndex([pd.Timestamp(t.year, t.month, t.day) for t in datas_cf])\n        alvo = origem + pd.offsets.MonthBegin(1)\n        exigir((datas < alvo).all(), f\'Initialization overlaps the target month: {origem.date()}.\')\n        exigir(datas.min() >= origem - pd.Timedelta(days=40) and datas.max() <= origem + pd.Timedelta(days=7), \'Unexpected pentad initialization window.\')\n        exigir(len(set(datas)) == n // 4, \'Four members per initialization date are required.\')\n        exigir(np.all(np.unique(datas.values, return_counts=True)[1] == 4), \'Pentad grouping mismatch.\')\n        resultado[origem] = dict(mascara=mascara, datas=datas, numeros=numeros, fase_fonte=\'hindcast\' if origem < pd.Timestamp(\'2011-04-01\') else \'operacional\')\n    return resultado\n\ndef carregar_membros(path, origens):\n    with xr.open_dataset(path, decode_times=False, engine=\'netcdf4\') as d:\n        ds = d.load()\n    exigir(set(ds.data_vars) == {\'prec\'}, \'The file must contain only predicted rainfall prec.\')\n    exigir(set(ds.prec.dims) == {\'S\', \'L\', \'M\', \'Y\', \'X\'}, \'Unexpected dimensions.\')\n    exigir(meses_de_S(ds.S).equals(pd.DatetimeIndex(origens)), \'NetCDF months differ from the request.\')\n    exigir(ds.sizes[\'L\'] == 1 and float(ds.L.values[0]) == LEAD, \'Incorrect lead; L=1.5 is required.\')\n    exigir(ds.L.attrs.get(\'units\') == \'months\' and float(ds.L.attrs.get(\'pointwidth\', -1)) == 1, \'A monthly forecast is required, not a multimonth average.\')\n    exigir(ds.prec.attrs.get(\'units\') == \'mm/day\', \'Unexpected units; no conversion will be assumed.\')\n    exigir(ds.prec.attrs.get(\'standard_name\') in (\'precipitation_rate\', \'lwe_precipitation_rate\'), \'Variable is not identified as a precipitation rate.\')\n    exigir(np.array_equal(ds.M.values, np.arange(1, 29)), \'Incomplete or duplicate member IDs.\')\n    exigir(np.array_equal(ds.Y.values, LAT) and np.array_equal(ds.X.values, LON_360), \'Native grid differs from the requested one-degree subset.\')\n    exigir(ds.X.attrs.get(\'units\') == \'degree_east\' and ds.Y.attrs.get(\'units\') == \'degree_north\', \'Geographic unit mismatch.\')\n    valores = ds.prec.transpose(\'S\', \'L\', \'M\', \'Y\', \'X\').values[:, 0].astype(np.float64)\n    marcador = ds.prec.attrs.get(\'file_missing_value\')\n    if marcador is not None and np.isfinite(float(marcador)):\n        valores[valores == float(marcador)] = np.nan\n    return valores\n\ndef membros_utilizaveis(membros, origem, info):\n    esperados = info[\'mascara\']\n    exigir(np.isnan(membros[~esperados]).all(), f\'A member outside the table contains data in {origem}.\')\n    completos = np.isfinite(membros).all(axis=(-1, -2))\n    faltantes = tuple((int(i + 1) for i in np.flatnonzero(esperados & ~completos)))\n    if faltantes:\n        exigir(faltantes == EXCECOES_MEMBROS.get(str(origem.date())), f\'Unexpected missing or partial members in {origem.date()}: {faltantes}.\')\n        exigir(np.isnan(membros[np.array(faltantes) - 1]).all(), \'The exception requires a completely absent member; partial gaps and infinities are rejected.\')\n    usados = esperados & completos\n    exigir((membros[usados] >= 0).all(), f\'Negative precipitation in {origem.date()}. Investigate before proceeding.\')\n    datas = info[\'datas\'][usados[esperados]]\n    return (usados, datas, faltantes)\n\ndef processar_bloco(path, origens, tabela):\n    origens = pd.DatetimeIndex(origens)\n    membros = carregar_membros(path, origens)\n    hash_bruto = sha256(path)\n    medias, linhas, datas_min, datas_max, ns, fases = ([], [], [], [], [], [])\n    for k, origem in enumerate(origens):\n        info = tabela[origem]\n        validos, datas, faltantes = membros_utilizaveis(membros[k], origem, info)\n        v = membros[k, validos]\n        exigir(np.isfinite(v).all(), f\'Missing or partial expected member in {origem}. Do not use a partial ensemble mean.\')\n        exigir((v >= 0).all(), f\'Negative precipitation in {origem}: minimum {v.min():.12g} mm/day. Investigate before proceeding.\')\n        media = v.mean(axis=0, dtype=np.float64).astype(np.float32)\n        alvo = origem + pd.offsets.MonthBegin(1)\n        medias.append(media)\n        ns.append(int(validos.sum()))\n        datas_min.append(datas.min())\n        datas_max.append(datas.max())\n        fases.append(0 if info[\'fase_fonte\'] == \'hindcast\' else 1)\n        linhas.append(dict(time_alvo=str(alvo.date()), time_origem=str(origem.date()), lead=LEAD, membros=ns[-1], membro_ids=\',\'.join(map(str, np.flatnonzero(validos) + 1)), membros_esperados=int(info[\'mascara\'].sum()), membros_indisponiveis_ids=list(faltantes), politica_membros=POLITICA_MEMBROS, inicializacao_mais_antiga=str(datas.min().date()), inicializacao_mais_recente=str(datas.max().date()), inicializacoes_anteriores_ao_alvo=True, fase_fonte=info[\'fase_fonte\'], minimo_membros_mm_day=float(v.min()), maximo_membros_mm_day=float(v.max()), minimo_media_mm_day=float(media.min()), maximo_media_mm_day=float(media.max()), media_espacial_mm_day=float(media.mean(dtype=np.float64)), membros_ausentes_estruturais=28 - int(info[\'mascara\'].sum()), pixels_ausentes_membros_validos=0, negativos_corrigidos=0, arquivo_bruto=path.name, sha256_bruto=hash_bruto))\n    ds = xr.Dataset({\'cfsv2_tp_media\': ((\'time\', \'lat\', \'lon\'), np.stack(medias)), \'n_membros\': (\'time\', np.array(ns, dtype=np.int16)), \'time_origem\': (\'time\', origens.values), \'inicializacao_mais_antiga\': (\'time\', pd.DatetimeIndex(datas_min).values), \'inicializacao_mais_recente\': (\'time\', pd.DatetimeIndex(datas_max).values), \'fase_fonte\': (\'time\', np.array(fases, dtype=np.int8))}, coords={\'time\': (origens.to_period(\'M\') + 1).to_timestamp(), \'lat\': LAT, \'lon\': LON_360 - 360}, attrs=dict(schema=SCHEMA, fonte=FONTE, modelo=\'NCEP-CFSv2\', amostragem=\'PENTAD_SAMPLES_FULL; 24 members, 28 in November\', produto=\'monthly forecast; local arithmetic mean of complete available members\', politica_membros=POLITICA_MEMBROS, lead_iri=LEAD, referencia_temporal=\'nominal S in the preceding month; actual initialization dates audited\', observacoes_de_chuva_utilizadas=\'nenhuma\', climatologia_aplicada=\'nenhuma\', grade=\'native one-degree grid; interpolation is performed in the modeling notebook\'))\n    ds.cfsv2_tp_media.attrs = dict(units=\'mm/day\', long_name=\'Predicted precipitation: CFSv2 ensemble mean\')\n    ds.lat.attrs[\'units\'], ds.lon.attrs[\'units\'] = (\'degrees_north\', \'degrees_east\')\n    ds.fase_fonte.attrs = dict(flag_values=np.array([0, 1], dtype=np.int8), flag_meanings=\'hindcast operacional\')\n    return (ds, linhas)\n', 'evaluate.py': '"""Verify frozen predictions without selecting models."""\nfrom pathlib import Path\nimport json\nimport tempfile\nfrom data_contracts import validar_previsao, metricas, conferir_observacao_bruta\nfrom registry import Registro, agora, canonico, deslocar, exigir, mes\n\ndef avaliar(registro, alvo, observacao, evidencia):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    exigir(agora() >= mes(deslocar(alvo, 4)), \'Primary verification requires T+4 and a final product.\')\n    p = r.protocolo()\n    previsoes = [e for e in r.eventos() if e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == alvo]\n    exigir(len(previsoes) == 1, \'No previously frozen prediction for this month.\')\n    previsao = previsoes[0]\n    exigir(evidencia.get(\'produto\') == \'ERA5 monthly_averaged_reanalysis\' and evidencia.get(\'expver\') == 1 and (evidencia.get(\'mes_alvo\') == alvo), \'Final-product metadata and the correct month are required.\')\n    exigir(evidencia.get(\'url\', \'\').startswith(\'https://cds.climate.copernicus.eu/\'), \'Unexpected verification source.\')\n    raw = Path(evidencia[\'arquivo_bruto\']).read_bytes()\n    auditoria = conferir_observacao_bruta(evidencia[\'arquivo_bruto\'], observacao, alvo)\n    with r.transacao(), tempfile.TemporaryDirectory() as td:\n        fp = Path(td) / \'previsao.nc\'\n        fp.write_bytes(r.ler_objeto(previsao[\'dados\'][\'previsao\']))\n        validar_previsao(fp, alvo)\n        linhas = metricas(fp, observacao)\n        anteriores = [e for e in r.eventos() if e[\'tipo\'] == \'verificacao\' and e[\'dados\'][\'mes_alvo\'] == alvo]\n        ob = r.guardar(Path(observacao).read_bytes())\n        bruto = r.guardar(raw)\n        ev = {k: v for k, v in evidencia.items() if k != \'arquivo_bruto\'}\n        ev[\'conferencia_bytes\'] = auditoria\n        return r._adicionar(\'verificacao\', dict(mes_alvo=alvo, protocolo=p[\'id\'], previsao=previsao[\'id\'], revisao=len(anteriores), primaria=not anteriores, metricas=linhas, evidencia=ev, observacao=ob, bruto=bruto, objetos=[ob, bruto]))\n\ndef painel(registro):\n    r = registro if isinstance(registro, Registro) else Registro(registro)\n    p = r.protocolo()[\'dados\'][\'plano\']\n    ev = r.eventos(conferir_objetos=True)\n    atual = p[\'primeiro_mes\']\n    rows = []\n    while atual <= p[\'ultimo_mes\']:\n        preds = [e for e in ev if e[\'tipo\'] == \'previsao_congelada\' and e[\'dados\'][\'mes_alvo\'] == atual]\n        obs = [e for e in ev if e[\'tipo\'] == \'verificacao\' and e[\'dados\'][\'mes_alvo\'] == atual and e[\'dados\'][\'primaria\']]\n        estado = \'avaliado\' if obs else \'emitido_aguardando_observacao\' if preds else \'nao_emitido_prazo_encerrado\' if agora() >= mes(atual) else \'aguardando_emissao\'\n        rows.append(dict(mes_alvo=atual, estado=estado, previsao=preds[0][\'id\'] if preds else None, verificacao=obs[0][\'id\'] if obs else None))\n        atual = deslocar(atual, 1)\n    resumo = dict(meses_planejados=len(rows), meses_emitidos=sum((x[\'previsao\'] is not None for x in rows)), meses_avaliados=sum((x[\'verificacao\'] is not None for x in rows)), meses=rows)\n    sums = {}\n    for e in ev:\n        if e[\'tipo\'] != \'verificacao\' or not e[\'dados\'][\'primaria\']:\n            continue\n        for a in e[\'dados\'][\'metricas\']:\n            chave = a[\'modelo\'] + \'/\' + a[\'regiao\']\n            z = sums.setdefault(chave, {k: 0 for k in [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']})\n            for k in z:\n                z[k] += a[k]\n    for z in sums.values():\n        z.update(rmse=(z[\'sse\'] / z[\'n\']) ** 0.5, mae=z[\'soma_erro_absoluto\'] / z[\'n\'], vies=z[\'soma_erro\'] / z[\'n\'], rmse_area=(z[\'sse_area\'] / z[\'peso_area\']) ** 0.5)\n    resumo[\'metricas_primarias\'] = sums\n    return resumo\n', 'test_registry.py': '"""Isolated tests: synthetic inputs never enter the real registry."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom unittest.mock import patch\nimport copy\nimport json\nimport tempfile\nimport unittest\nimport numpy as np\nimport xarray as xr\nfrom data_contracts import LAT, LON, indices_psl\nfrom registry import Registro, ContratoError, digest, deslocar\nfrom evaluate import painel, avaliar\nBASE = Path(__file__).resolve().parent\nNOW = datetime(2026, 9, 28, 20, 0, tzinfo=timezone.utc)\n\nclass TestRegistro(unittest.TestCase):\n\n    def setUp(self):\n        self.temp = tempfile.TemporaryDirectory()\n        self.addCleanup(self.temp.cleanup)\n        self.raiz = Path(self.temp.name)\n        self.clock = patch(\'registry.agora\', return_value=NOW)\n        self.clock.start()\n        self.addCleanup(self.clock.stop)\n        self.clock2 = patch(\'evaluate.agora\', return_value=NOW)\n        self.clock2.start()\n        self.addCleanup(self.clock2.stop)\n        self.r = Registro(self.raiz / \'registro\')\n        self.plano = json.loads((BASE / \'plan.json\').read_text())\n        self.protocolo = self.r.iniciar(self.plano)\n\n    def pronto(self):\n        arquivos = {}\n        for nome in self.plano[\'arquivos_modelo\']:\n            p = self.raiz / nome\n            p.write_bytes(b\'fixture apenas sintetica\')\n            arquivos[nome] = p\n        self.r.modelo(arquivos, dict(treino_inicio=\'1993-01\', treino_fim=\'2022-12\', protocolo_id=self.protocolo[\'id\']))\n        for n, regra in self.plano[\'fontes\'].items():\n            if regra[\'obrigatoria\']:\n                self.r.recibo(n, b\'fixture sintetica \' + n.encode(), dict(validado=True, meses=[deslocar(\'2026-10\', -regra[\'lag\'])]), {\'url\': \'fixture\'})\n        self.fp = self.raiz / \'previsao.nc\'\n        a = np.ones((1, len(LAT), len(LON)), dtype=np.float32)\n        ds = xr.Dataset({n: ((\'time\', \'lat\', \'lon\'), a, {\'units\': \'mm/day\'}) for n in [\'precipitacao\', \'climatologia\']}, coords={\'time\': [np.datetime64(\'2026-10-01\')], \'lat\': LAT, \'lon\': LON})\n        ds.to_netcdf(self.fp)\n        s = self.r.prontidao(\'2026-10\')\n        self.inf = {k: s[k] for k in [\'modelo\', \'fontes\', \'protocolo\']}\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n\n    def test_coleta_e_integridade(self):\n        e = self.r.recibo(\'nino34\', b\'abc\', dict(validado=False, meses=[]), {\'url\': \'fixture\'})\n        self.assertEqual(self.r.ler_objeto(e[\'dados\'][\'objeto\']), b\'abc\')\n        p = self.r.pasta / \'objetos\' / (e[\'dados\'][\'objeto\'][\'sha256\'] + \'.bin\')\n        p.write_bytes(b\'abd\')\n        with self.assertRaises(ContratoError):\n            self.r.eventos(conferir_objetos=True)\n\n    def test_cadeia_detecta_evento_editado(self):\n        self.r.adicionar(\'coleta_falhou\', {\'objetos\': []})\n        p = sorted((self.r.pasta / \'eventos\').glob(\'*.json\'))[-1]\n        e = json.loads(p.read_text())\n        e[\'tipo\'] = \'fonte_recebida\'\n        p.write_text(json.dumps(e))\n        with self.assertRaises(ContratoError):\n            self.r.eventos()\n\n    def test_protocolo_imutavel(self):\n        self.r.iniciar(self.plano)\n        outro = copy.deepcopy(self.plano)\n        outro[\'modelo\'][\'arvores\'] = 900\n        with self.assertRaises(ContratoError):\n            self.r.iniciar(outro)\n\n    def test_faltantes_bloqueiam(self):\n        s = self.r.prontidao(\'2026-10\')\n        self.assertFalse(s[\'pronto\'])\n        self.assertEqual(len(s[\'faltantes\']), 9)\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', \'inexistente\', {})\n\n    def test_nao_aceita_mes_errado_ou_quarentena(self):\n        self.r.recibo(\'nino34\', b\'abc\', dict(validado=True, meses=[\'2026-08\']), {\'url\': \'fixture\'})\n        self.r.recibo(\'nino12\', b\'abc\', dict(validado=False, meses=[\'2026-07\']), {\'url\': \'fixture\'})\n        self.assertNotIn(\'nino34\', self.r.prontidao(\'2026-10\')[\'fontes\'])\n        self.assertNotIn(\'nino12\', self.r.prontidao(\'2026-10\')[\'fontes\'])\n\n    def test_congelamento_e_reemissao_bloqueada(self):\n        self.pronto()\n        self.assertFalse(self.r.prontidao(\'2026-10\')[\'previsao_emitida\'])\n        e = self.r.congelar(\'2026-10\', self.fp, self.inf)\n        self.assertEqual(e[\'registrado_em_utc\'], NOW.isoformat())\n        self.assertEqual(e[\'dados\'][\'validacao\'][\'pontos\'], 78561)\n        self.assertTrue(self.r.prontidao(\'2026-10\')[\'previsao_emitida\'])\n        self.assertFalse(self.r.prontidao(\'2026-11\')[\'previsao_emitida\'])\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_prazo_nao_pode_ser_retroagido(self):\n        self.pronto()\n        with patch(\'registry.agora\', return_value=datetime(2026, 10, 1, tzinfo=timezone.utc)):\n            with self.assertRaises(ContratoError):\n                self.r.congelar(\'2026-10\', self.fp, self.inf)\n        self.assertFalse(any((e[\'tipo\'] == \'previsao_congelada\' for e in self.r.eventos())))\n\n    def test_troca_fonte_apos_inferencia_bloqueia(self):\n        self.pronto()\n        self.r.recibo(\'tna\', b\'outra versao\', dict(validado=True, meses=[\'2026-07\']), {\'url\': \'fixture\'})\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_grid_e_valores_errados_bloqueiam(self):\n        self.pronto()\n        with xr.open_dataset(self.fp) as raw:\n            ds = raw.load()\n        ds[\'precipitacao\'].values[0, 0, 0] = np.nan\n        ds.to_netcdf(self.fp, mode=\'w\')\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n        ds[\'precipitacao\'].values[0, 0, 0] = 1\n        ds = ds.assign_coords(lon=LON + 0.01)\n        ds.to_netcdf(self.fp, mode=\'w\')\n        self.inf[\'previsao_sha256\'] = digest(self.fp.read_bytes())\n        with self.assertRaises(ContratoError):\n            self.r.congelar(\'2026-10\', self.fp, self.inf)\n\n    def test_painel_inclui_meses_sem_emissao(self):\n        d = painel(self.r)\n        self.assertEqual(d[\'meses_planejados\'], 12)\n        self.assertEqual(d[\'meses_emitidos\'], 0)\n        self.assertEqual(d[\'metricas_primarias\'], {})\n        with patch(\'evaluate.agora\', return_value=datetime(2026, 11, 2, tzinfo=timezone.utc)):\n            self.assertEqual(painel(self.r)[\'meses\'][0][\'estado\'], \'nao_emitido_prazo_encerrado\')\n\n    def test_avaliacao_nao_libera_alvo_antes_da_hora(self):\n        self.pronto()\n        self.r.congelar(\'2026-10\', self.fp, self.inf)\n        with self.assertRaises(ContratoError):\n            avaliar(self.r, \'2026-10\', \'inexistente\', {})\n\n    def test_avaliacao_primaria_nao_sobrescrita_por_revisao(self):\n        self.pronto()\n        self.r.congelar(\'2026-10\', self.fp, self.inf)\n        with xr.open_dataset(self.fp) as pred:\n            truth = pred[[\'precipitacao\']].rename({\'precipitacao\': \'tp\'}).load()\n        obs = self.raiz / \'obs.nc\'\n        truth.to_netcdf(obs)\n        raw = truth.copy(deep=True)\n        raw.tp.values[:] = 0.001\n        raw.tp.attrs = dict(units=\'m\', GRIB_stream=\'moda\', GRIB_experimentVersionNumber=\'0001\')\n        bruto = self.raiz / \'bruto.nc\'\n        raw.to_netcdf(bruto)\n        info = dict(produto=\'ERA5 monthly_averaged_reanalysis\', expver=1, mes_alvo=\'2026-10\', url=\'https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means\', arquivo_bruto=str(bruto))\n        mature = datetime(2027, 2, 2, tzinfo=timezone.utc)\n        with patch(\'registry.agora\', return_value=mature), patch(\'evaluate.agora\', return_value=mature):\n            prim = avaliar(self.r, \'2026-10\', obs, info)\n            self.assertTrue(prim[\'dados\'][\'primaria\'])\n            truth.tp.values[:] = 2\n            truth.to_netcdf(obs, mode=\'w\')\n            with self.assertRaises(ContratoError):\n                avaliar(self.r, \'2026-10\', obs, info)\n            raw.tp.values[:] = 0.002\n            raw.to_netcdf(bruto, mode=\'w\')\n            rev = avaliar(self.r, \'2026-10\', obs, info)\n            self.assertFalse(rev[\'dados\'][\'primaria\'])\n            self.assertEqual(painel(self.r)[\'metricas_primarias\'][\'precipitacao/dominio\'][\'rmse\'], 0)\n\n    def test_derivado_exige_pais_da_mesma_fonte(self):\n        e = self.r.recibo(\'nino34\', b\'abc\', dict(validado=False, meses=[]), {\'url\': \'fixture\'})\n        codigo = self.raiz / \'adapter.py\'\n        codigo.write_text(\'fixture\')\n        with self.assertRaises(ContratoError):\n            self.r.derivado(\'cfsv2\', b\'def\', dict(validado=True, meses=[\'2026-09\']), [e[\'id\']], codigo)\n\n    def test_parser_psl_nao_preenche_ausentes(self):\n        b = (\'2026 2026\\n2026 \' + \' \'.join([\'1\'] * 7 + [\'-99.99\'] * 5) + \'\\n-99.99\\nHadISST\\n\').encode()\n        d = indices_psl(b)\n        self.assertEqual(d[\'ultimo_mes\'], \'2026-07\')\n        self.assertEqual(d[\'ausentes\'], 5)\n        with self.assertRaises(ContratoError):\n            indices_psl(b\'<html>login</html>\')\nif __name__ == \'__main__\':\n    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestRegistro)\n    resultado = unittest.TextTestRunner(verbosity=2).run(suite)\n    if not resultado.wasSuccessful():\n        raise SystemExit(1)\n', 'plan.json': '{\n  "id": "nimbus_acompanhamento_controle_defasado_v1",\n  "equipe": "Nimbus PA",\n  "primeiro_mes": "2026-10",\n  "ultimo_mes": "2027-09",\n  "prazo": "antes de 00:00 UTC do primeiro dia do mes alvo; sem emissao retroativa",\n  "treino_inicio": "1993-01",\n  "treino_fim": "2022-12",\n  "politica_modelo": "um pacote final do controle com defasagens, reajustado na janela fixa; sem SST/PCA; sem atualizar pesos ou treino durante estes doze meses",\n  "modelo": {\n    "arvores": 300,\n    "folhas": 31,\n    "taxa": 0.05,\n    "semente": 42,\n    "pontos_por_mes": 5000,\n    "ridge_alpha": 0.1,\n    "pesos": [0.375, 0.375, 0.25],\n    "escalas_residuais": [0.9, 0.875],\n    "maximo_meses_treino": 360\n  },\n  "arquivos_modelo": [\n    "arvores_seas5.txt", "arvores_multissistema.txt", "ridge.npz",\n    "clima_chuva.nc", "clima_atmosfera_indices.npz", "clima_seas5.nc",\n    "clima_cfsv2.nc", "biblioteca.py", "inferir.py", "ambiente.json",\n    "auditoria_treino.json"\n  ],\n  "fontes": {\n    "era5_sl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal; tcc,t2m,sp"},\n    "era5_pl": {"lag": 4, "obrigatoria": true, "produto": "ERA5 final mensal 850 hPa; z,r,q,t,u,v"},\n    "nino34": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino34.long.anom.data"},\n    "nino12": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL nino12.long.anom.data"},\n    "tna": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tna.data"},\n    "tsa": {"lag": 3, "obrigatoria": true, "produto": "NOAA PSL tsa.data"},\n    "seas5": {"lag": 1, "obrigatoria": true, "produto": "SEAS5 sistema 51, inicializacao T-1, forecastMonth 2; datas e membros auditados"},\n    "cfsv2": {"lag": 1, "obrigatoria": true, "produto": "CFSv2 IRI PENTAD_SAMPLES_FULL; emissao nominal T-1 e lead 1.5; membros auditados"},\n    "ersstv5": {"lag": 2, "obrigatoria": false, "produto": "ERSSTv5; somente arquivo de chegada, nao entra na referencia"}\n  },\n  "falha_de_fonte": "bloquear emissao; nao preencher, trocar mes, versao ou modelo silenciosamente",\n  "selecao_versao": "ultima versao recebida e validada antes da emissao; registrar IDs exatos",\n  "avaliacao": {\n    "alvo": "ERA5 mensal final total_precipitation em mm/day; nao representa pluviometro independente",\n    "primeira_verificacao": "primeira versao final conferida recebida a partir do primeiro dia de T+4",\n    "metrica_primaria": "RMSE global sqrt(soma SSE / soma n)",\n    "secundarias": ["MAE", "vies", "RMSE por area", "tres faixas de latitude", "meses emitidos/previstos"],\n    "comparacao": "climatologia do mesmo treino congelada junto a cada previsao",\n    "revisoes": "adicionar outra verificacao rotulada; nunca sobrescrever a primeira",\n    "faltantes": "reportar cobertura; nao omitir meses sem emissao dos relatorios"\n  },\n  "agendamento_automatico": false,\n  "carimbo_independente": false,\n  "limitacao": "Registro local com relogio UTC do computador e hashes. Sem assinatura/ancoragem externa; nao prova contra reescrita integral ou exclusao da ponta. Qualidade e calendario de cada adaptador devem ser auditados."\n}\n', 'requirements.txt': 'numpy>=2.0,<3\npandas>=2.2,<4\nxarray>=2025.1\nrequests>=2.31,<3\nnetCDF4>=1.6,<2\ncftime>=1.6,<2\nnbformat>=5.10,<6\n'}
BUNDLE_SHA256 = {'README.md': 'fa8d6f8a63b99ebd9ed412861b783c3c8c7b75209c78fa0190089e3bad5e50c5', 'registry.py': '71a2d34a21c5f1005aa424f3465e23f963ec92f06491cd184c605c7d9f036690', 'data_contracts.py': '8986792dd070fb1a040c969f47498015e06bae9f91ca6eeaa3018d4ca23ded9f', 'collect_sources.py': '1b86c81e84070b26b0c322bef3f727e2774717914616cd092cfecfd88810c270', 'prepare_cfsv2.py': '962ca1b8661f7492084aba8896577cfe43d3375bbd43e3469367404f4ae2da10', 'cfsv2_adapter.py': '79c698b6e0cd1cb16aa21f0b919bc521e4836cdb0252a605f3a7268eb24e136b', 'evaluate.py': '0e1297cb9acfb22005fb22e6ea2054e484899933a15a3d349a43dd2d978a20a8', 'test_registry.py': 'bbac9ddaca1c377f14a157432a07f3d66a101680d7c335dc903ab8897183bc0d', 'plan.json': '7e1ab0eb8aa5fefaddfbf5a3a04a907a1c77bb7cfee6e4a47a639e6adfb04359', 'requirements.txt': '5d94e4558764ac18a61f9198c5636813ca14b4ddd57a5718a23d3da386e786f9'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODIGO/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('This directory contains a different source version. Preserve it and choose a new output directory.')
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODIGO))
print('Source bundle verified:', CODIGO)


## 1. Restore the previous registry
Attach the last saved output to preserve the chain across Kaggle sessions. Never replace receipt timestamps with earlier dates.

In [ ]:
from registry import Registro
REGISTRO = RAIZ/'registry'
# Resume only a verified event chain and its content-addressed objects.
if not (REGISTRO/'eventos').exists() and Path('/kaggle/input').is_dir():
    previous = [p.parent for p in Path('/kaggle/input').rglob('resumo_registro.json')
                if (p.parent/'eventos').is_dir() and (p.parent/'objetos').is_dir()]
    if len(previous) > 1:
        raise RuntimeError('Attach exactly one previous registry to resume.')
    if previous:
        Registro(previous[0]).eventos(conferir_objetos=True)
        REGISTRO.mkdir(parents=True, exist_ok=True)
        for folder in ['eventos','objetos']:
            shutil.copytree(previous[0]/folder, REGISTRO/folder)
        print('Previous registry verified and restored. Receipt times are unchanged.')
print('Registry prepared. No forecast has been issued.')


## 2. Run isolated tests
Synthetic inputs are created in temporary directories. They never enter the real registry.

In [ ]:
runpy.run_path(str(CODIGO/'test_registry.py'),run_name='__main__')


## 3. Collect public sources and audit CFSv2
Enable Internet. These downloads require no token. Responses are recorded at the actual receipt time. Missing ensemble members block preparation; the code does not substitute another month or product. Review the target month before running.

In [ ]:
from collect_sources import executar
from prepare_cfsv2 import preparar
ALVO = '2026-10'
ESTADO = executar(REGISTRO, CODIGO/'plan.json', ALVO, incluir_cfsv2=True)
CFS_RECEIPT = preparar(REGISTRO, ALVO)


## 4. Inspect readiness and preserve outputs
This notebook records arrivals and validates readiness. Fitting and emission are implemented in the separate baseline_inference.ipynb workflow. No forecast is emitted by this source-collection notebook.

In [ ]:
from evaluate import painel
from IPython.display import display, Markdown, FileLink
r = Registro(REGISTRO)
r.exportar_resumo(REGISTRO/'resumo_registro.json')
state = r.prontidao(ALVO)
(REGISTRO/'prontidao.json').write_text(json.dumps(state,indent=2,ensure_ascii=False),encoding='utf-8')
(REGISTRO/'painel.json').write_text(json.dumps(painel(r),indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(state,indent=2,ensure_ascii=False))
display(Markdown('**Save the version with outputs.** The registry records when each input arrived. This run does not fit a model or issue a forecast.'))
display(FileLink(str(REGISTRO/'resumo_registro.json')))
